# S1–S6 density-amplitude growth

Growth of the per-cycle density-perturbation peaks with altitude, normalised to the base cell (nearest 150 km), for the azimuthal-resolution series S1–S6.

* Heavy part: `compute_growth_cache.py --background azimean_preforcing --altitude-cells nearest` (CLI, run in the background) writes `outputs/s1_s6_azimuthal/cache/density_growth_<run>_azimean_preforcing_nearest.pkl`. This notebook only reads those files. `cached_growth` computes a missing cache, which loads GB-scale fields.
* **Recipe** (`dsmclib.resolution.STAGING_RECIPE`):
  * The background is averaged over all azimuth columns and the 17 pre-forcing outputs common to every run (t = 4908–5012 s).
  * The growth window runs over the cells nearest 150 and 320 km, i.e. 148.3–315.4 km.
  * The root `optimized_loader` background (`mad_column0`, azimuth column 0 only) carries several % of particle noise per altitude, which made the S1–S3 profiles crooked above ~270 km. To see that, set `RECIPE = {"background": "mad_column0", "altitude_cells": "first_above"}`, or see `peak_diagnostics.ipynb`.
* Method: `dsmclib.growth`. With the root recipe it is bit-identical to the root code (`checks/regress_density_growth.py`). With `azimean_snapshot` + `nearest` + 600 s padding it reproduces the 2024 growth pkls (`checks/regress_legacy_2024_growth.py`).
* S1–S3 are 10-member ensemble means (see the registry notes).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from dsmclib import registry, plotting
from dsmclib.growth import cached_growth, growth_curve, growth_matrix
from dsmclib.paths import output_dir

plotting.use_style(font_scale=1.4)
CAMPAIGN = "s1_s6_azimuthal"
FIG_DIR = output_dir(CAMPAIGN, "figures")
runs = registry.campaign(CAMPAIGN)   # include_alternates=True adds S4_2025
runs

[Run(S1: Results_20240719_003802.440, campaign=s1_s6_azimuthal, role=current, n_azi=128, periods=5),
 Run(S2: Results_20240719_025013.785, campaign=s1_s6_azimuthal, role=current, n_azi=256, periods=5),
 Run(S3: Results_20240719_025200.987, campaign=s1_s6_azimuthal, role=current, n_azi=384, periods=5),
 Run(S4: Results_20240722_034301.982, campaign=s1_s6_azimuthal, role=current, n_azi=512, periods=5),
 Run(S5: Results_20240722_040346.456, campaign=s1_s6_azimuthal, role=current, n_azi=640, periods=5),
 Run(S6: Results_20240722_041906.318, campaign=s1_s6_azimuthal, role=current, n_azi=1024, periods=5)]

In [2]:
from dsmclib.resolution import STAGING_RECIPE

RECIPE = STAGING_RECIPE   # or {"background": "mad_column0", "altitude_cells": "first_above"} (root optimized_loader)
growth = {r.alias: cached_growth(r.alias, **RECIPE) for r in runs}
{alias: (len(g["azimuthal_indices"]), g["alt_range"]) for alias, g in growth.items()}

{'S1': (12, (150.0, 320.0)),
 'S2': (23, (150.0, 320.0)),
 'S3': (34, (150.0, 320.0)),
 'S4': (44, (150.0, 320.0)),
 'S5': (55, (150.0, 320.0)),
 'S6': (88, (150.0, 320.0))}

## Centre vs edge of the forcing footprint

Centre = the middle forced column. Edge = the mean of the first and last forced columns.

In [3]:
def centre_and_edge(g, n_last_peaks=5):
    cols = np.asarray(g["azimuthal_indices"])
    alt, centre = growth_curve(g, cols[len(cols) // 2], n_last_peaks)
    _, left = growth_curve(g, cols[0], n_last_peaks)
    _, right = growth_curve(g, cols[-1], n_last_peaks)
    return alt, np.asarray(centre), 0.5 * (np.asarray(left) + np.asarray(right))

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(runs)))
for color, r in zip(colors, runs):
    alt, centre, edge = centre_and_edge(growth[r.alias])
    axes[0].plot(centre, alt, color=color, marker="o", ms=3, label=f"{r.alias} ({r.n_azi_cell} azi)")
    axes[1].plot(edge, alt, color=color, marker="o", ms=3, label=r.alias)
for ax, title in zip(axes, ["Centre column", "Footprint edges"]):
    ax.plot(plotting.TIAN_GROW, plotting.TIAN_ALTITUDES, "k--", label="Tian et al. (2023)")
    ax.plot(plotting.SPHERICAL_1D_GROW, plotting.SPHERICAL_1D_ALTITUDES, "k:", label="Spherical 1D")
    ax.set_title(title)
    ax.set_xlabel("Density amplitude growth")
axes[0].set_ylabel("Altitude (km)")
axes[0].legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / "growth_centre_vs_edge_S1-S6.png", dpi=200, bbox_inches="tight")

/Users/josepc/miniforge/envs/dsmc/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


## Growth across the footprint (one run)

In [4]:
alias = "S6"
ax = plotting.plot_growth_heatmap(growth[alias])
ax.figure.savefig(FIG_DIR / f"growth_heatmap_{alias}.png", dpi=200, bbox_inches="tight")

In [5]:
ax = plotting.plot_growth_curves(growth[alias])
ax.figure.savefig(FIG_DIR / f"growth_all_columns_{alias}.png", dpi=200, bbox_inches="tight")